`aix` is a python package that facades several AI tools. 
This notebook demos some of its functionalities.

In [ ]:
from contaix

# AI chat

`aix.chat_funcs` gathers the models you have access to in your environment 
(depends on what you have installed (e.g. `google.generativeai`, `oa` (itself a facade to `openai` functionality), etc.)).

In [1]:
from aix import chat_funcs

list(chat_funcs)

['gpt-4', 'gpt-4-32k', 'gpt-4-turbo', 'o1', 'o1-mini', 'gpt-4o', 'gpt-4o-mini']

`chat_funcs` is a dictionary whos keys are the names of the available models, and 
values are a `chat` function with the model set to that model name. 

In [3]:
chat_funcs['o1-mini']

functools.partial(<function chat at 0x1355f68c0>, model='o1-mini')

Note that different providers have different interfaces, but the functions that 
`chat_funcs` provides all have `prompt` as their first argument. 

In [8]:
from inspect import signature

signature(chat_funcs['o1-mini'])

<Sig (prompt=None, *, model='o1-mini', messages=None, frequency_penalty: 'Optional[float] | NotGiven' = NOT_GIVEN, function_call: 'completion_create_params.FunctionCall | NotGiven' = NOT_GIVEN, functions: 'Iterable[completion_create_params.Function] | NotGiven' = NOT_GIVEN, logit_bias: 'Optional[Dict[str, int]] | NotGiven' = NOT_GIVEN, logprobs: 'Optional[bool] | NotGiven' = NOT_GIVEN, max_tokens: 'Optional[int] | NotGiven' = NOT_GIVEN, n: 'Optional[int] | NotGiven' = NOT_GIVEN, parallel_tool_calls: 'bool | NotGiven' = NOT_GIVEN, presence_penalty: 'Optional[float] | NotGiven' = NOT_GIVEN, response_format: 'completion_create_params.ResponseFormat | NotGiven' = NOT_GIVEN, seed: 'Optional[int] | NotGiven' = NOT_GIVEN, service_tier: "Optional[Literal['auto', 'default']] | NotGiven" = NOT_GIVEN, stop: 'Union[Optional[str], List[str]] | NotGiven' = NOT_GIVEN, stream: 'Optional[Literal[False]] | Literal[True] | NotGiven' = NOT_GIVEN, stream_options: 'Optional[ChatCompletionStreamOptionsParam]

In [9]:
signature(chat_funcs['gemini-1.5-flash'])

<Sig (prompt: str, *, model='gemini-1.5-flash', generation_config: 'generation_types.GenerationConfigType | None' = None, safety_settings: 'safety_types.SafetySettingOptions | None' = None, stream: 'bool' = False, tools: 'content_types.FunctionLibraryType | None' = None, tool_config: 'content_types.ToolConfigType | None' = None, request_options: 'helper_types.RequestOptionsType | None' = None)>

For tab-completion convenience, the (python identifier version of the) models 
were placed as attributes of `chat_funcs`, so you can access them directly there.

In [ ]:
print(chat_funcs.gemini_1_5_flash('What is the capital of France?'))

The capital of France is **Paris**. 



In [ ]:
print(chat_funcs.gpt_3_5_turbo('What is the capital of France?'))

The capital of France is Paris.


There's also a dictionary called `chat_models` that contains the same keys:

In [2]:
from aix import chat_models

list(chat_models)

['gpt-4', 'gpt-4-32k', 'gpt-4-turbo', 'o1', 'o1-mini', 'gpt-4o', 'gpt-4o-mini']

But here the values are some useful metadatas on the model, like pricing...

In [6]:
chat_models['gpt-4o']

{'price_per_million_tokens': 5.0,
 'pages_per_dollar': 804,
 'performance_on_eval': 'Efficiency-optimized version of GPT-4 for better performance on reasoning tasks',
 'max_input': 8192,
 'provider': 'openai'}

You can can a dataframe with all the info for all the models like this:

In [5]:
import pandas as pd 
pd.DataFrame(index=chat_models.keys(), data=chat_models.values())

,price_per_million_tokens,price_per_million_tokens_output,pages_per_dollar,performance_on_eval,max_input,provider
gpt-4,30.00,60.0,134,Advanced reasoning for complex tasks,8192,openai
gpt-4-32k,60.00,120.0,67,Extended context window for long documents,32768,openai
gpt-4-turbo,10.00,30.0,402,Cost-effective version of GPT-4,8192,openai
o1,15.00,60.0,268,Optimized for complex reasoning in STEM fields,8192,openai
o1-mini,1.10,4.4,1341,Cost-effective reasoning for simpler tasks,8192,openai
gpt-4o,2.50,10.0,804,Efficiency-optimized version of GPT-4 for bett...,8192,openai
gpt-4o-mini,0.15,0.6,13410,"Highly cost-effective, optimized for simple ta...",8192,openai


The corresponding attributes are only the name of the model (the key itself):

In [5]:
chat_models.gpt_4

'gpt-4'

This is for the convenience of entering a model name in a different context, with 
less errors than if you were typing the name as a string. 
For example, you can enter it yourself in the general `chat` function:

In [13]:
from aix import chat, chat_models

chat('How many Rs in "Strawberry"?', model=chat_models.gpt_4o, frequency_penalty=0.5)  

'The word "Strawberry" contains two instances of the letter \'R\'.'

# More models

In [22]:
from aix import models, chat

# Discover models from OpenRouter (includes DeepSeek, Claude, etc.)
model_infos = models.discover('openrouter')

In [23]:
len(model_infos)

342

In [24]:
model_info = model_infos[0]
type(model_info)

aix.ai_models.base.Model

In [25]:
list(model_info)

['id',
 'provider',
 'context_size',
 'is_local',
 'capabilities',
 'cost_per_token',
 'tags',
 'connector_metadata',
 'custom_metadata']

In [26]:
model_info['id']

'nvidia/nemotron-3-nano-30b-a3b:free'

In [ ]:
import tabled, pandas as pd

model_info_table = tabled.expand_columns(pd.DataFrame(model_infos), 'cost_per_token')
# move cost_per_token.input	and cost_per_token.output in position 1 and 2
model_info_table

,id,provider,context_size,is_local,capabilities,tags,connector_metadata,custom_metadata,cost_per_token.input,cost_per_token.output
0,nvidia/nemotron-3-nano-30b-a3b:free,,256000,False,{},{},{'openrouter': {'id': 'nvidia/nemotron-3-nano-...,{},0.000000e+00,0.000000e+00
1,openai/gpt-5.2-chat,,128000,False,{},{},"{'openrouter': {'id': 'openai/gpt-5.2-chat', '...",{},1.750000e-06,1.400000e-05
2,openai/gpt-5.2-pro,,400000,False,{},{},"{'openrouter': {'id': 'openai/gpt-5.2-pro', 'n...",{},2.100000e-05,1.680000e-04
3,openai/gpt-5.2,,400000,False,{},{},"{'openrouter': {'id': 'openai/gpt-5.2', 'name'...",{},1.750000e-06,1.400000e-05
4,mistralai/devstral-2512:free,,262144,False,{},{},{'openrouter': {'id': 'mistralai/devstral-2512...,{},0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...
337,undi95/remm-slerp-l2-13b,,6144,False,{},{},{'openrouter': {'id': 'undi95/remm-slerp-l2-13...,{},4.500000e-07,6.500000e-07
338,gryphe/mythomax-l2-13b,,4096,False,{},{},{'openrouter': {'id': 'gryphe/mythomax-l2-13b'...,{},6.000000e-08,6.000000e-08
339,openai/gpt-4-0314,,8191,False,{},{},"{'openrouter': {'id': 'openai/gpt-4-0314', 'na...",{},3.000000e-05,6.000000e-05
340,openai/gpt-3.5-turbo,,16385,False,{},{},"{'openrouter': {'id': 'openai/gpt-3.5-turbo', ...",{},5.000000e-07,1.500000e-06


Now look at THOSE choices!! Yum!

In [16]:
chat("Hello", model="openrouter/deepseek/deepseek-chat")

'Hello!  😊 How can I assist you today?'

In [ ]:
chat("Hello", model="openai/gpt-5.2-chat")


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



NotFoundError: litellm.NotFoundError: OpenAIException - The model `gpt-5.2-chat` does not exist or you do not have access to it.

In [ ]:
chat("Hello", model="openrouter/openai/gpt-5.2-chat")
# 


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



APIError: litellm.APIError: APIError: OpenrouterException - {"error":{"message":"This request requires more credits, or fewer max_tokens. You requested up to 16384 tokens, but can only afford 2856. To increase, visit https://openrouter.ai/settings/keys and create a key with a higher total limit","code":402,"metadata":{"provider_name":null}},"user_id":"user_32E4idAKom3deyxBG7Ouqib9xtb"}